# 08 - Summarizing and combining

Last time we got a file off the disk and into a table we could trust. We ended with a function that
does it in one line - and with an admission that the table still is not right, because 43 of its
entities are not countries at all.

This notebook is about the two things that fix that, and they are the two most useful operations in
all of data analysis:

- **Summarizing**: answering a question about *groups* of rows rather than single ones - `groupby`,
  aggregating, and transforming
- **Combining**: putting two tables together - `concat` and `merge`, and how to check that the merge
  did what you think it did

Along the way: what happens when the group is a date, how to measure change from one row to the next,
and what to do about duplicate rows.

By the end we can answer the question we could not answer last time: **what were the world's total
CO₂ emissions in 2023?**

> 📝 **Note:** A few cells in this notebook are *meant* to fail, and are marked with a comment
naming the error, for example `# MergeError`. Several other cells produce answers that are **wrong
without failing** - those are the point of the section they are in, and the text says so. Run cells
one at a time rather than using Run All.

As always, we start by importing what we need and loading our own data.

In [1]:
import numpy as np
import pandas as pd

And here is the function we finished with last time. Nothing in it is new: it reads the file, drops
the rows with no emissions figure, and adds emissions per person.

In [2]:
def load_emissions(path):
    """
    Read the emissions file and return it ready to use.

    Parameters
    ----------
    path : str
        Path to the emissions CSV file.

    Returns
    -------
    DataFrame
        One row per country per year, with rows missing total emissions
        dropped and emissions per person added as a column.
    """
    emissions = pd.read_csv(path)

    emissions = emissions.dropna(subset=["co2_total"])
    emissions["co2_pc"] = emissions["co2_total"] * 1_000_000 / emissions["population"]

    return emissions


co2 = load_emissions("../data/co2_emissions.csv")

print(co2.shape)
co2.head(3)

(5904, 12)


,country,code,year,co2_total,population,urban,gdp_pc,electricity,agriculture,nat_resources,renew_energy,co2_pc
0,Afghanistan,AFG,2000,1.0093,20130327,18.558,174.93,4.4,57.946,NaN,45.0,0.050138
1,Afghanistan,AFG,2001,0.9402,20284307,18.741,138.71,9.3,57.947,NaN,45.6,0.046351
2,Afghanistan,AFG,2002,0.9388,21378117,18.941,178.95,14.1,57.940,1.2761,37.8,0.043914


In [5]:
co2["code"].unique()

<StringArray>
['AFG', 'AFE', 'AFW', 'ALB', 'DZA', 'ASM', 'AGO', 'ATG', 'ARB', 'ARG',
 ...
 'URY', 'UZB', 'VUT', 'VEN', 'VNM', 'VIR', 'WLD', 'YEM', 'ZMB', 'ZWE']
Length: 246, dtype: str

## Splitting the table into groups

Here is a question the table looks like it can answer. **How much CO₂ did the world emit each
year?**

Every row is one entity in one year. So: take all the rows for a year, add up their emissions, and
do that for each year. That is a loop over years, and you could write one - you have written harder
loops than that. pandas has a shorter way, and it is called `groupby`.

(`.tail(3)` below shows the **last** three rows, the way `.head(3)` shows the first three.)

In [3]:
co2.groupby("year")["co2_total"].sum().tail(3)

year
2021    312230.8520
2022    314199.0783
2023    320155.6020
Name: co2_total, dtype: float64

Three moves in one line: **split** the rows into groups by their `year`, **apply** `sum` to the
`co2_total` column of each group, and **combine** the answers into one result. That is the whole
idea, and it has a name - **split-apply-combine**. Every `groupby` you will ever write is those
three steps, and the only things that change are what you split on and what you apply.

So the world emitted about **320 156** million tonnes of CO₂ in 2023. That is the answer, and it
is wrong.

Here is how we know. The dataset contains a row called `World`, which is the World Bank's own figure
for the same quantity.

In [11]:
co2[(co2["country"] == "World") & (co2["year"] == 2023)][["country", "year", "co2_total"]]

,country,year,co2_total
6167,World,2023,39112.6888


**39 113 against our 320 156.** We are out by a factor of more than eight, with no error and no
warning anywhere.

Nothing went wrong with `groupby`. It added up exactly the rows we gave it - and we gave it rows like
`World`, `Arab World` and `East Asia & Pacific`, which are groupings of countries. China is in the
table once as China, again inside `East Asia & Pacific`, again inside `Middle income`, again inside
`World`. Summing them all counts the same smokestack over and over.

We cannot fix this yet, because the emissions file does not say which entities are countries and
which are not. That information is in a second file, and joining two files together is the second
half of this notebook. **Hold on to the number 320 156. We come back for it.**

### The grouped object computes nothing

It is worth seeing what `groupby` actually returns, because it is not a table.

In [12]:
by_year = co2.groupby("year")

by_year

A `DataFrameGroupBy`. It knows which rows belong to which group and it has computed nothing at all -
it is waiting to be told what to apply. That is why the two halves are always written together:
`.groupby("year")` on its own is a question with no verb.

Pick a column first and you get one answer per group.

In [16]:
co2.groupby("year")["co2_pc"].mean().tail(3)

year
2021    4.525511
2022    4.495202
2023    4.406028
Name: co2_pc, dtype: float64

Leave the column out and pandas applies the function to every numeric column it can, which is
occasionally what you want and usually noise.

### Counting: `.size()` and `.count()`

Two ways to count, and the difference matters.

- `.size()` counts **rows** in each group.
- `.count()` counts **non-missing values** in a column of each group.

In [17]:
counts = pd.DataFrame({
    "rows": co2.groupby("year").size(),
    "renew_energy": co2.groupby("year")["renew_energy"].count(),
})

counts.tail(5)

,rows,renew_energy
year,,
2019,246,246
2020,246,246
2021,246,203
2022,246,66
2023,246,0


Every year has the same 246 rows, and until 2020 every row has a renewable energy figure. Then 203,
then 66, then **none at all**. That is the publication lag we found last time, seen from a different
direction: the rows are all there, the values are not.

Reach for `.size()` when you want to know how big a group is, and `.count()` when you want to know
how much of it you can actually use.

> 💡 **Tip:** You can group on more than one thing at a time by passing a **list** of column
names. We do not have a sensible second column to group on yet - the one we want says which region
each country is in, and it lives in a file we have not opened. We come back to it.

<div class="alert alert-info" style="background-color:#d9edf7; border-left:6px solid #31708f; border-radius:4px; padding:12px 16px; color:#31708f;">
<h3 style="margin-top:0; color:#31708f;">Your turn</h3>

<p>Starting from:</p>

<pre style="background-color:#ffffff; color:#31708f; padding:8px 10px; border-radius:3px;"><code>co2 = pd.read_csv("../data/co2_emissions.csv")</code></pre>

<p>For each year, report the <b>highest</b> value of <code>gdp_pc</code> in that year, and display
the last five years.</p>
</div>

## Several answers at once: `.agg()`

`.mean()` gives you one statistic. Real summary tables want several, and writing one `groupby` per
statistic and gluing the results together is miserable. `.agg()` takes a **list of function names,
written as strings**, and gives you a column for each.

In [18]:
co2.groupby("year")["co2_pc"].agg(["mean", "median", "max"]).tail(3)

,mean,median,max
year,,,
2021,4.525511,2.609076,74.734297
2022,4.495202,2.643982,78.726280
2023,4.406028,2.686580,78.857111


The names are the same ones you already use as methods - `"mean"`, `"median"`, `"min"`, `"max"`,
`"sum"`, `"std"`, `"count"`, `"size"`, `"first"`, `"last"`. Inside `.agg()` they are written as
strings; on their own they are written with brackets.

That works when every statistic is of the **same column**. More often you want different statistics
of different columns, and then there is a second form: give each output column a name, and say what
it is made of.

```python
grouped.agg(
    output_name = ("input_column", "function"),
    ...
)
```

In [21]:
summary = co2.groupby("year").agg(
    entities=("country", "count"),
    total_co2=("co2_total", "sum"),
    mean_pc=("co2_pc", "mean"),
)

summary.tail(3)

,entities,total_co2,mean_pc
year,,,
2021,246,312230.8520,4.525511
2022,246,314199.0783,4.495202
2023,246,320155.6020,4.406028


One line per output column, each saying exactly where it came from. This is the form to reach for
when you are building a table somebody else will read, because the column names are yours rather
than pandas' guesses.

> 💡 **Tip:** `.round(2)` on the result rounds every number in it, which usually makes a
summary table far easier to read. Round for **display**, at the end - not in the middle of a
calculation, where you would be throwing away precision you still need.

<div class="alert alert-info" style="background-color:#d9edf7; border-left:6px solid #31708f; border-radius:4px; padding:12px 16px; color:#31708f;">
<h3 style="margin-top:0; color:#31708f;">Your turn</h3>

<p>Starting from:</p>

<pre style="background-color:#ffffff; color:#31708f; padding:8px 10px; border-radius:3px;"><code>co2 = pd.read_csv("../data/co2_emissions.csv")</code></pre>

<p>Build one summary table with a row per year and three named columns: the number of entities that
have a <code>gdp_pc</code> figure, the average <code>gdp_pc</code>, and the highest
<code>urban</code> share. Display the last five years, rounded to one decimal.</p>
</div>

## What a grouped result *is*

Look closely at what came back from those last few cells. The years are not in a column. They are
down the left-hand side, where the row numbers usually go.

That is because they **are** the index now. `groupby` puts whatever you grouped on into the index of
the result, which makes sense: the group label is what identifies the row.

In [22]:
totals = co2.groupby("year")["co2_total"].sum()

print(type(totals))
print(totals.index)

<class 'pandas.Series'>
Index([2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011,
       2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023],
      dtype='int64', name='year')


A `Series`, indexed by year. That is often exactly what you want. Index alignment means you can
divide one grouped result by another and the years line themselves up, and plotting a Series puts the
index along the horizontal axis without being asked.

But it is a nuisance the moment you want to treat the result as an ordinary table - filter it, join
it, or write it to a file - because **`year` is not a column**, and asking for it as one fails.

In [23]:
# KeyError
totals["year"]

KeyError: 'year'

### `reset_index` moves the index back into a column

In [24]:
totals_df = totals.reset_index()

print(type(totals_df))
totals_df.tail(3)

<class 'pandas.DataFrame'>


,year,co2_total
21,2021,312230.8520
22,2022,314199.0783
23,2023,320155.6020


Now it is a DataFrame with two ordinary columns, and everything from last time works on it again.
`reset_index()` is the most common thing to write after a `groupby`, and when a grouped result is
fighting you, it is usually the answer.

### `set_index` goes the other way

`set_index` takes a column and makes it the index.

In [25]:
totals_df.set_index("year").tail(3)

,co2_total
year,
2021,312230.8520
2022,314199.0783
2023,320155.6020


Which raises the obvious question: why would you ever do that on purpose?

Because an index is not decoration - it is what pandas **aligns on**, as we saw when two Series with
different labels were added together. A table indexed by something meaningful can be looked up by
label with `.loc`, lines up automatically with any other table indexed the same way, and - as we are
about to see - can be grouped by time.

In [30]:
totals_by_year = totals_df.set_index("year")

totals_by_year.loc[2023]

co2_total    320155.602
Name: 2023, dtype: float64

> 📝 **Note:** `reset_index()` takes a `drop` parameter. `reset_index(drop=True)` throws the
old index away instead of turning it into a column, which is what you want when the index is
meaningless row numbers rather than something you care about. It turns up later in this notebook.

## Same answer, same shape: `.transform()`

Everything so far **collapsed** the table: 5 904 rows in, 24 rows out, one per group.

Often that is not what you want. You want the group's answer put back **beside every row it came
from**, so that a row can be compared against its own group. Norway's emissions per person in 2020
are only interesting next to Norway's usual level.

`.transform()` does exactly that. Same groups, same function names - but the result has **the same
number of rows as the table you started with**.

In [31]:
print("mean      :", co2.groupby("country")["co2_pc"].mean().shape)
print("transform :", co2.groupby("country")["co2_pc"].transform("mean").shape)

mean      : (246,)
transform : (5904,)


246 against 5 904. `.mean()` gave one number per country; `.transform("mean")` gave every row its own
country's number, repeated as many times as that country has rows.

Because it comes back the same length as the table, it can be assigned straight into a column - which
is the whole point of it.

In [32]:
co2["country_mean_pc"] = co2.groupby("country")["co2_pc"].transform("mean")
co2["pc_vs_own_mean"] = co2["co2_pc"] - co2["country_mean_pc"]

co2[co2["country"] == "Norway"][["year", "co2_pc", "country_mean_pc", "pc_vs_own_mean"]].tail(6)

,year,co2_pc,country_mean_pc,pc_vs_own_mean
4290,2018,8.780335,9.09245,-0.312115
4291,2019,8.479335,9.09245,-0.613115
4292,2020,8.124938,9.09245,-0.967512
4293,2021,8.214714,9.09245,-0.877736
4294,2022,7.979858,9.09245,-1.112592
4295,2023,7.537538,9.09245,-1.554912


Norway's own 24-year average is 9.09 tonnes per person, and every year since 2018 has come in below
it by a widening margin. Notice what this is *not*: it is not a comparison against other countries.
Each row is measured against its own history, which is very often the fairer question.

This pattern - **compute a baseline per group, then measure every row against it** - is one of the
most useful things in the course. Temperature anomalies, deviations from trend, shares of a total and
index numbers are all these same two lines.

> ⚠️ **Warning:** `mean` and `transform("mean")` differ in *shape*, not in difficulty. If you assign
a grouped result to a column and get a wall of `NaN`, you almost certainly used an aggregation where
you needed a transformation: pandas aligned 246 country names against 5 904 row numbers, found no
matches, and filled the lot with missing values. That is index alignment doing exactly what it
promised to do.

### Filling gaps within a group

Two more methods belong to this family and are useful enough to name. **`ffill`** carries the last
known value *forward* into the holes after it, and **`bfill`** carries the next known value
*backward* into the holes before it.

We know `renew_energy` stops being published in the most recent years. A forward fill would carry
each country's last observed share forward into them.

In [ ]:
filled = co2.groupby("country")["renew_energy"].ffill()

print("missing before:", co2["renew_energy"].isna().sum())
print("missing after: ", filled.isna().sum())

471 holes down to 2. Here is what it did to one country.

In [33]:
co2["renew_filled"] = co2.groupby("country")["renew_energy"].ffill()

co2[co2["country"] == "Norway"][["year", "renew_energy", "renew_filled"]].tail(5)

,year,renew_energy,renew_filled
4291,2019,59.8,59.8
4292,2020,60.9,60.9
4293,2021,61.4,61.4
4294,2022,NaN,61.4
4295,2023,NaN,61.4


Norway's 2021 figure of 61.4 has been copied into 2022 and into 2023.

**And now the part you have to decide rather than run.** A forward fill asserts that the value did
not change. Over one year, for a share that moves slowly, that is defensible. Over three years, for
exactly the years everyone wants to look at, it invents a flat line that never happened - and nothing
downstream will ever tell you the number was made up. Filling is a claim about the world, not a
tidying step.

`bfill` is the same method pointing the other way, and it earns its keep when a label is recorded
only once. If a table gives each country's region in its final year only, one `bfill` within country
carries it back over every earlier year.

### The group is not optional

Now the same fill, without the `groupby`.

In [ ]:
print("grouped ffill,   missing after:", co2.groupby("country")["renew_energy"].ffill().isna().sum())
print("ungrouped ffill, missing after:", co2["renew_energy"].ffill().isna().sum())

**Zero.** The ungrouped version looks better and is very much worse. Having run out of Norwegian
values it carried straight on into the next country's rows and filled those with Norway's number. The
table is sorted by country, so the last Norwegian figure became the first figure of whichever country
follows.

No error, no warning: a column with nothing missing in it, and some of the numbers belong to the
wrong country.

> ⚠️ **Warning:** Any method that looks at the row before or the row after - `ffill`, `bfill`,
`diff`, `shift`, `pct_change` - has no idea where one country ends and the next begins. **Group
first**, and the group is whatever the rows are a series *within*.

<div class="alert alert-info" style="background-color:#d9edf7; border-left:6px solid #31708f; border-radius:4px; padding:12px 16px; color:#31708f;">
<h3 style="margin-top:0; color:#31708f;">Your turn</h3>

<p>Starting from:</p>

<pre style="background-color:#ffffff; color:#31708f; padding:8px 10px; border-radius:3px;"><code>co2 = pd.read_csv("../data/co2_emissions.csv")
co2["co2_pc"] = co2["co2_total"] * 1_000_000 / co2["population"]</code></pre>

<p>Add a column holding each country's <b>highest</b> <code>co2_pc</code> across all years, then
display only the rows where a country reached that maximum - so that each country appears in the
year that was its worst.</p>
</div>

## When the group is a date

A great deal of real data is a **time series**: the same thing measured over and over, with a date
attached. Grouping such data by month or by year is one of the commonest things anybody does with
it - and to do that, pandas has to know that the date column is a date.

New dataset, and a small one: the daily share price of Apple through 2020.

In [ ]:
apple = pd.read_csv("../data/AAPL.csv")

print(apple.shape)
apple.head(3)

252 rows - one per trading day - and seven columns. `Open`, `High`, `Low` and `Close` are prices
through the day, and `Volume` is the number of shares traded.

Now the important part. What type is `Date`?

In [ ]:
apple.dtypes

`str`. As far as pandas is concerned, `"2020-01-02"` is a piece of text like `"Bergen"`. You can sort
it and compare it, and both will *appear* to work, because a date written year-month-day happens to
sort correctly as text. That is luck, and it runs out the moment the file uses `02/01/2020` instead.

`pd.to_datetime` converts a column of text into real dates.

In [ ]:
apple["Date"] = pd.to_datetime(apple["Date"])

print(apple["Date"].dtype)
apple["Date"].head(3)

`datetime64` - a point in time, which pandas can do arithmetic on.

> ⚠️ **Warning:** If your dates are not written year-first, say so with `format=`. Given
`"02/01/2021"`, `pd.to_datetime` assumes the American month-first reading and returns **1 February**,
not 2 January - and because both readings are valid calendar dates, there is no error. Pass
`format="%d/%m/%Y"` to be certain (`%d` day, `%m` month, `%Y` four-digit year), and always eyeball a
few converted values against the originals.

### `.dt` gets the pieces out

A real date knows what year, month and day it is made of. The `.dt` accessor is how you ask.

In [ ]:
apple["month"] = apple["Date"].dt.month

apple[["Date", "month", "Close"]].head(3)

`.dt.year`, `.dt.month`, `.dt.day`, `.dt.quarter` and `.dt.day_name()` all work the same way. And
because `month` is now an ordinary column, everything from the first half of this notebook applies to
it - `groupby` included.

In [ ]:
apple.groupby("month")["Close"].mean().head(4)

Apple's average closing price fell from 78 in January to 66 in March - the pandemic crash, showing
up in the data.

Real dates also compare properly, so filtering to a period is a mask like any other - except that you
can write the boundaries as strings and pandas will read them as dates.

In [ ]:
march = apple[(apple["Date"] >= "2020-03-01") & (apple["Date"] <= "2020-03-31")]

print(len(march), "trading days in March")

<div class="alert alert-info" style="background-color:#d9edf7; border-left:6px solid #31708f; border-radius:4px; padding:12px 16px; color:#31708f;">
<h3 style="margin-top:0; color:#31708f;">Your turn</h3>

<p>Starting from:</p>

<pre style="background-color:#ffffff; color:#31708f; padding:8px 10px; border-radius:3px;"><code>apple = pd.read_csv("../data/AAPL.csv")
apple["Date"] = pd.to_datetime(apple["Date"])</code></pre>

<p>Which month of 2020 had the highest total trading <code>Volume</code>? Display the months ordered
from busiest to quietest.</p>
</div>

## `resample`: grouping on the calendar

There is a second way to group a time series, and it exists because `groupby` gets something subtly
wrong.

`resample` works like `groupby`, but the groups are **periods of time** - months, quarters, years,
weeks. It requires the dates to be the **index**, which is what `set_index` was for, and it wants
them in order, which is what `sort_index` is for.

In [ ]:
apple_dated = apple.set_index("Date").sort_index()

apple_dated["Close"].resample("ME").mean().head(3)

`"ME"` means month-end. `"QE"` is quarter-end, `"YE"` year-end, `"W"` weekly, `"D"` daily.

Compare that against the `groupby` from a moment ago and you will notice something unhelpful: the
numbers are **identical**. 77.98, 77.82, 65.61 either way.

In [ ]:
print("groupby  rows:", len(apple.groupby("month")["Close"].mean()))
print("resample rows:", len(apple_dated["Close"].resample("ME").mean()))

Twelve and twelve. On this dataset the two are interchangeable, so it is completely unclear why
`resample` should exist at all.

That is because this dataset is **one year long**, which hides the difference entirely. Here is a
table small enough to see whole, covering two.

In [ ]:
sales = pd.DataFrame({
    "date": pd.to_datetime([
        "2020-01-15", "2020-02-10", "2020-12-20",
        "2021-01-11", "2021-02-03", "2021-12-14",
    ]),
    "sales": [100, 120, 140, 160, 180, 200],
})

sales

In [ ]:
sales.groupby(sales["date"].dt.month)["sales"].sum()

In [ ]:
sales.set_index("date")["sales"].resample("ME").sum()

Six rows in. **Three rows out one way and twenty-four out the other**, and both are correct answers
to different questions.

Two things happened, and each of them is the whole point:

1. **`groupby` merged January 2020 with January 2021.** `.dt.month` returns the number 1 for both, so
   they are the same group. That is right if you want a seasonal pattern - "how do Januaries look?" -
   and catastrophically wrong if you want a time series.
2. **`resample` produced months that contain no rows at all**, showing `0` for March through November
   in both years. `groupby` cannot do that: it only knows about groups that have rows in them.

The rule underneath both:

> **`groupby` builds its groups out of the rows you have. `resample` builds its groups out of the
> calendar.**

So `resample` is the one to reach for whenever the answer should be a series through time - because a
month with no observations is *information*, and silently omitting it is what makes a chart lie.

That second property is easiest to see by asking for something finer than the data.

In [ ]:
daily = apple_dated["Close"].resample("D").mean()

print("trading days in the file:", len(apple_dated))
print("calendar days produced:  ", len(daily))
print("of which empty:          ", daily.isna().sum())

364 rows from 252, with 112 of them empty. Those are the weekends and public holidays: days that
exist in the calendar and on which no shares were traded. `resample` made them visible rather than
pretending the year had 252 days in it.

> 💡 **Tip:** `resample` is `groupby` for time and takes the same verbs after it - `.mean()`,
`.sum()`, `.last()`, `.count()`, `.agg()`. `resample("W").last()` gives the closing price at the end
of each week, which is how weekly price series are almost always built.

## Change from one row to the next: `.diff()`

A series of levels usually is not the question. The question is what changed.

`.diff()` subtracts each row from the one after it.

In [ ]:
apple_dated["change"] = apple_dated["Close"].diff()

apple_dated[["Close", "change"]].head(4)

The first row is `NaN`, because there is no day before it to subtract - which is honest, and is why
the column is a float.

So what was the worst day of 2020 to own this share?

In [ ]:
apple_dated["change"].sort_values().head(3)

3 September, down 10.52. Now ask the same question in percentage terms - `.pct_change()` is `.diff()`
divided by the previous value.

In [ ]:
(apple_dated["Close"].pct_change() * 100).sort_values().head(3)

**A different day wins: 16 March, down 12.86%.** Neither answer is wrong. The share was worth around
68 in March and around 120 in September, so the same percentage move is a much bigger number of
dollars later in the year. Which one you want depends on the question, and the two disagreeing is
worth knowing before you report one of them.

### And on a panel, the group comes back

Our emissions table is 246 separate series stacked on top of each other. Sort it by country and year
and the rows *look* like a time series - so `.diff()` will happily run on it.

In [ ]:
panel = co2.sort_values(["country", "year"]).copy()

panel["change_wrong"] = panel["co2_total"].diff()
panel["change"] = panel.groupby("country")["co2_total"].diff()

panel[panel["change"].isna() & panel["change_wrong"].notna()][
    ["country", "year", "co2_total", "change_wrong", "change"]
].head(3)

These are the rows where the two disagree, and every one of them is a country's **first** year. The
grouped version says `NaN` - correctly, there is no 1999 to compare against. The ungrouped version
reports a number, and that number is this country's first year minus the *previous country's* last
year. Albania emitted 3.23 million tonnes in 2000, and the table records that as a **fall of 248.41**
- which is not a fact about Albania.

In [ ]:
print("rows where the ungrouped version is wrong:", (panel["change"].isna() & panel["change_wrong"].notna()).sum())

245 of them - one per country, minus the first. Under one row in twenty, scattered through the table,
each one a plausible-looking number in the wrong place. This is the same bug as the ungrouped
`ffill`, wearing a different hat, and `.notna()` above is just `.isna()` the other way round.

With the grouped version we can ask something worth asking.

In [ ]:
year_2020 = panel[panel["year"] == 2020]

print("entities with a 2020 figure:", len(year_2020))
print("emissions fell in 2020:     ", (year_2020["change"] < 0).sum())

187 of the 246 entities emitted less CO₂ in 2020 than in 2019. That is not a subtle effect and it
took one grouped `.diff()` to find - but only because the `groupby` was there. Without it, 245 of
those comparisons would have been against the wrong country.

<div class="alert alert-info" style="background-color:#d9edf7; border-left:6px solid #31708f; border-radius:4px; padding:12px 16px; color:#31708f;">
<h3 style="margin-top:0; color:#31708f;">Your turn</h3>

<p>Starting from:</p>

<pre style="background-color:#ffffff; color:#31708f; padding:8px 10px; border-radius:3px;"><code>co2 = pd.read_csv("../data/co2_emissions.csv")
co2 = co2.sort_values(["country", "year"])</code></pre>

<p>Add a column giving each country's year-on-year <b>percentage</b> change in
<code>co2_total</code>, and display the five largest single-year rises in the whole table.</p>
</div>

## Stacking tables: `concat`

Half of this notebook done, and every table has been one file. Real work is rarely one file: a year
per file, a country per file, one file from each of ten colleagues.

When the tables have the **same columns and different rows**, you stack them. `pd.concat` takes a
**list** of DataFrames and returns one.

In [ ]:
north = pd.DataFrame({
    "country": ["Norway", "Sweden", "Denmark"],
    "co2_pc": [7.5, 3.4, 4.6],
})

south = pd.DataFrame({
    "country": ["Spain", "Italy"],
    "co2_pc": [5.0, 5.4],
})

stacked = pd.concat([north, south])

stacked

Five rows, as expected. But look down the left-hand side: **0, 1, 2, 0, 1**. `concat` stacked the
indexes too, so two different rows are both labeled 0.

That is not cosmetic. The index is what pandas aligns on, so a duplicated index quietly breaks `.loc`
and anything built on it.

In [ ]:
stacked.loc[0]

Two rows returned for one label. The fix is the method from earlier, with the `drop` parameter -
these row numbers mean nothing, so throw them away rather than keeping them as a column.

In [ ]:
stacked = pd.concat([north, south]).reset_index(drop=True)

stacked

**Write it as one expression, every time.** `pd.concat([...]).reset_index(drop=True)` is the idiom;
the version without it is a bug waiting for somebody to use `.loc`.

If the tables do not have quite the same columns, `concat` keeps all of them and fills the gaps with
`NaN` rather than refusing.

In [ ]:
extra = pd.DataFrame({
    "country": ["Poland"],
    "co2_pc": [8.0],
    "note": ["provisional"],
})

pd.concat([north, extra]).reset_index(drop=True)

Convenient, and worth checking rather than trusting: a column of `NaN` you did not expect usually
means the two files disagree about a column name - `co2_pc` in one and `CO2_pc` in the other - and
`concat` will not tell you.

> 📝 **Note:** `pd.concat([a, b], axis=1)` glues tables side by side instead, matching on the
index. It is occasionally right and usually not: when you want to add *columns* from another table,
what you almost always want is a merge, which is the next section.

## Duplicates

Stacking files together is the commonest way to end up with the same row twice - a file listed twice
in a folder, a cell run twice, two colleagues who both sent you January.

`.duplicated()` gives one `True` or `False` per row, marking rows identical to one seen earlier. It
is a boolean mask, so summing it counts them.

In [ ]:
readings = pd.DataFrame({
    "station": ["Oslo", "Bergen", "Oslo", "Tromsø", "Bergen"],
    "year": [2023, 2023, 2023, 2023, 2023],
    "temp": [6.9, 8.4, 6.9, 3.1, 8.4],
})

print("duplicate rows:", readings.duplicated().sum())
readings

Rows 2 and 4 repeat rows 0 and 1 exactly. `.drop_duplicates()` removes them, keeping the first
occurrence.

In [ ]:
readings.drop_duplicates()

That was the easy case, and it is easy because the duplicate rows **agree**. Nothing is lost by
dropping one.

Here is the case that matters.

In [ ]:
readings_2 = pd.DataFrame({
    "station": ["Oslo", "Bergen", "Oslo", "Tromsø"],
    "year": [2023, 2023, 2023, 2023],
    "temp": [6.9, 8.4, 7.4, 3.1],
})

print("duplicate rows:", readings_2.duplicated().sum())
readings_2

**Zero duplicates**, and Oslo is still in there twice - with two different temperatures. No row is a
copy of another, so `.duplicated()` is right to say nothing.

To ask "is any station listed twice?", say which columns identify a row, using `subset`.

In [ ]:
print("repeated station-year keys:", readings_2.duplicated(subset=["station", "year"]).sum())

readings_2[readings_2.duplicated(subset=["station", "year"], keep=False)]

`keep=False` marks **every** row involved rather than only the later one, which is what you want when
you are looking at the problem instead of deleting it.

And you should be looking at it, because this is not a cleaning problem.

> ⚠️ **Warning:** An exactly duplicated row is a mistake, and dropping it loses nothing. A repeated
**key** with conflicting values is a question about your data - which figure is right, and why are
there two? `drop_duplicates(subset=["station", "year"])` would silently keep 6.9 and discard 7.4 for
no better reason than that it came first. Find out before you drop.

<div class="alert alert-info" style="background-color:#d9edf7; border-left:6px solid #31708f; border-radius:4px; padding:12px 16px; color:#31708f;">
<h3 style="margin-top:0; color:#31708f;">Your turn</h3>

<p>Starting from:</p>

<pre style="background-color:#ffffff; color:#31708f; padding:8px 10px; border-radius:3px;"><code>info = pd.read_csv("../data/country_info.csv")</code></pre>

<p>This lookup table should have exactly one row per country. Check both things that could be wrong
with it: how many rows are exact duplicates, and how many <code>code</code> values appear more than
once. Print both counts.</p>
</div>

## Merging: columns from another table

`concat` adds rows. **`merge` adds columns**, by matching rows in one table against rows in another
on a shared value called a **key**.

This is what we have been waiting for. Here is the second file.

In [ ]:
info = pd.read_csv("../data/country_info.csv")

print(info.shape)
info.head(3)

295 rows, one per entity: a `name`, a three-letter `code`, a `region` and an `incomeLevel`. And the
column that fixes our 320 156 problem is `region`, because the World Bank marks its own groupings
with the region `"Aggregates"`.

In [ ]:
info["region"].value_counts()

### First, look at the keys

Both tables have something identifying an entity - `country` and `code` in one, `name` and `code` in
the other. So there are two possible keys, and they are not equally good.

A merge matches keys **exactly**. Not roughly, not case-insensitively: exactly. And text keys arrive
dirty far more often than anybody expects, so it is worth two minutes looking at them first.

Every string method from the first half of this course - `.strip()`, `.lower()`, `.replace()`,
`.startswith()` - is available on a whole column at once through the **`.str` accessor**. It is
vectorized, exactly like the arithmetic on columns we did last time: no loop, one expression.

In [ ]:
print(info["name"].str.lower().head(3))
print()
print("names with something to strip:", (info["name"] != info["name"].str.strip()).sum())

**Four of the 295 names have leading or trailing whitespace**, exactly as the World Bank publishes
them. `"Sub-Saharan Africa "` and `"Sub-Saharan Africa"` look identical on screen and are two
different strings, so a key built on `name` will fail on those four and succeed everywhere else -
which is the worst possible behavior, because it is small enough to miss.

`.str.strip()` would fix them. `code` does not have the problem at all. Both facts matter, and we
will use both.

### Merging

`.merge()` is called on one table and given the other, plus `on=` naming the shared column.

In [ ]:
panel = co2.merge(info[["code", "region", "incomeLevel"]], on="code", how="left")

print("before:", co2.shape)
print("after: ", panel.shape)
panel[["country", "year", "co2_total", "region", "incomeLevel"]].head(3)

Same number of rows, two extra columns. `how="left"` says *keep every row of the left-hand table,
whether or not it found a match*, and it is the right default for this job: we are decorating the
emissions data, not filtering it.

The four kinds of join differ only in which unmatched rows survive:

| `how=` | Keeps |
|---|---|
| `"inner"` | only rows that matched on **both** sides - the default, and it deletes silently |
| `"left"` | every row of the left table; unmatched right-hand columns become `NaN` |
| `"right"` | every row of the right table |
| `"outer"` | everything from both sides |

Prefer `"left"` when one table is your data and the other is a lookup. Prefer `"inner"` only when you
genuinely want the intersection - and even then, count what it cost you.

### A merge can make your table bigger

The one behavior that surprises people. If a key appears **more than once** in the right-hand table,
every match produces a row.

In [ ]:
countries = pd.DataFrame({
    "code": ["NOR", "SWE"],
    "name": ["Norway", "Sweden"],
})

visits = pd.DataFrame({
    "code": ["NOR", "NOR", "NOR", "SWE"],
    "year": [2021, 2022, 2023, 2023],
})

countries.merge(visits, on="code", how="left")

Two rows in, four rows out, from a **left** join. Nothing is wrong: Norway genuinely has three
matching rows. But if you expected one row per country and got this, every average you compute
afterwards is now weighted by how many times each country happened to appear.

**A merge can lose rows and a merge can gain rows, and neither one raises anything.** So you check.

## Checking that the merge did what you think

Three checks, in increasing order of how much work they save you.

### 1. Count the rows

The cheapest thing in pandas, and the one that catches most of it. Below we deliberately merge on the
dirty key instead, to see what it costs - and because the key is called `country` in one table and
`name` in the other, we name each side separately with `left_on` and `right_on` rather than `on`.

In [ ]:
merged_on_name = co2.merge(
    info[["name", "region"]],
    left_on="country",
    right_on="name",
    how="inner",
)

print("rows before:", len(co2))
print("rows after: ", len(merged_on_name))
print("lost:       ", len(co2) - len(merged_on_name))

**48 rows gone.** That is the `name` key meeting the trailing spaces, and an inner join deleting
whatever failed to match without comment. 48 out of 5 904 is under one percent - far too small to
notice in a `.head()`, more than enough to be wrong about.

### 2. `indicator=True` says which side each row came from

Counting tells you *that* something went missing. This tells you *what*.

In [ ]:
checked = co2.merge(
    info[["name", "region"]],
    left_on="country",
    right_on="name",
    how="left",
    indicator=True,
)

checked["_merge"].value_counts()

Done as a **left** join with `indicator=True`, nothing is deleted and a column called `_merge` says
what happened to each row: `both` if it matched, `left_only` if it did not. 5 856 matched, 48 did
not.

This is the diagnostic pattern worth remembering: **when a merge surprises you, redo it as a left
join with `indicator=True` and look at the rows that failed.**

In [ ]:
unmatched = co2[~co2["country"].isin(info["name"])]["country"].unique()

print(list(unmatched))

Two entity names, 24 years each, 48 rows. And they are obviously present in the other file - you can
see `Sub-Saharan Africa` in it with your own eyes. Print the other table's version with `repr`, which
shows a string the way Python would write it, quotes and all.

In [ ]:
print([repr(name) for name in info[info["name"].str.strip().isin(unmatched)]["name"]])

There it is: `'Sub-Saharan Africa '`. A single trailing space, invisible in every display pandas ever
produced, and it cost 48 rows.

Two fixes, and both are one line: clean the key with `.str.strip()`, or use a key that was never
dirty. Here `code` is right there, so we use it - but on a file with no code column, `.str.strip()`
on both sides before merging is the standard defense.

### 3. `validate=` refuses to run a merge you did not mean

The first two checks happen *after* the damage. `validate=` happens before: you state what shape the
merge is supposed to be, and pandas raises if it is not.

- `"one_to_one"` - the key is unique in both tables
- `"many_to_one"` - repeated on the left, unique on the right; the usual shape for a lookup table
- `"one_to_many"` - the other way round

Our merge is many-to-one: 24 rows per country on the left, one row per country on the right.

In [ ]:
panel = co2.merge(
    info[["code", "region", "incomeLevel"]],
    on="code",
    how="left",
    validate="many_to_one",
)

print("validated, rows:", len(panel))

Silence, which means it held. Claim the wrong shape and it does not.

In [ ]:
# MergeError
co2.merge(info[["code", "region"]], on="code", how="left", validate="one_to_one")

`Merge keys are not unique in left dataset` - because of course they are not, there are 24 rows per
country. That is the error doing its job: it caught a false belief about the data before any number
came out of it.

> 💡 **Tip:** Put `validate=` on every merge you write. It costs one argument and turns a
whole class of silent wrongness into an exception, which is the best trade available in pandas.

### One more silent one, and it is `groupby`

We now have a `region` column, so we can finally group on two things at once - and this is the moment
to notice something `groupby` does that nobody mentions.

Take the merge that failed, where 48 rows have no region, and total emissions by region for 2023.

In [ ]:
year_2023 = checked[checked["year"] == 2023]

by_region = year_2023.groupby("region")["co2_total"].sum()

print("sum of the grouped result:", by_region.sum().round(1))
print("sum of the column:        ", year_2023["co2_total"].sum().round(1))

**2 619 million tonnes have vanished between one line and the next.**

`groupby` silently drops every row whose grouping key is missing. There is a defensible reason -
there is no group called "missing" to put them in - but the effect is that a broken merge and a
`groupby` combine into a total that is quietly too small, with nothing anywhere saying so.

> ⚠️ **Warning:** After grouping, check that the parts add up to the whole. If a key can be missing,
`groupby(..., dropna=False)` keeps those rows in a group of their own rather than deleting them.

### The payoff

We have everything we need. Drop the aggregates and ask the question again.

In [ ]:
countries_only = panel[panel["region"] != "Aggregates"]

print("all entities:", panel["code"].nunique(), "->", countries_only["code"].nunique(), "countries")
print("rows:        ", len(panel), "->", len(countries_only))

In [ ]:
total_2023 = countries_only[countries_only["year"] == 2023]["co2_total"].sum()
world_2023 = panel[(panel["country"] == "World") & (panel["year"] == 2023)]["co2_total"].iloc[0]

print(f"sum over 203 countries: {total_2023:10,.0f}")
print(f"the World Bank's World: {world_2023:10,.0f}")

**37 483 against 39 113** - about four percent short, instead of out by a factor of eight.

The gap that remains is real and explicable: 43 entities were aggregates, 14 more countries have no
emissions figure at all and were dropped when we loaded the file, and the World Bank's `World` row
includes territories our 203 do not. A number that is close but not identical, for reasons you can
name, is a much better place to be than a number that matches exactly by luck.

And now the two-key `groupby` we could not write earlier.

In [ ]:
countries_only.groupby(["region", "year"])["co2_total"].sum().head(4)

Two labels down the left-hand side instead of one. That is a **MultiIndex** - the same idea as
before, with the group being a *pair* rather than a single value. It is worth knowing that it exists;
it is usually not worth wrestling with, and `reset_index()` turns it into two ordinary columns, which
is what most code wants.

In [ ]:
regional = countries_only.groupby(["region", "year"])["co2_total"].sum().reset_index()

regional[regional["year"] == 2023].round(0)

<div class="alert alert-info" style="background-color:#d9edf7; border-left:6px solid #31708f; border-radius:4px; padding:12px 16px; color:#31708f;">
<h3 style="margin-top:0; color:#31708f;">Your turn</h3>

<p>Starting from:</p>

<pre style="background-color:#ffffff; color:#31708f; padding:8px 10px; border-radius:3px;"><code>co2 = pd.read_csv("../data/co2_emissions.csv")
info = pd.read_csv("../data/country_info.csv")</code></pre>

<p>Merge the two tables on <code>code</code> with a left join and <code>validate=</code> set to the
right shape, drop the aggregates, and display the ten countries with the highest total emissions in
2023. Print the row count before and after the merge.</p>
</div>

## A function of your own, applied to each group

Almost everything in this notebook has been a pandas method with a name in quotes: `"mean"`, `"sum"`,
`"max"`. Sooner or later a group needs an answer that is not on that list.

Here is one. What are a region's emissions **per person**? Not the average of its countries' figures -
that would let Luxembourg count as much as India. The region's total emissions divided by the
region's total population. Every built-in aggregation works on **one column**; this one needs two at
once, so none of them can express it.

Last time we said `.apply()` on a column - one value in, one value out, once per row - is usually the
wrong shape, and that vectorized alternatives are shorter and faster. That argument stands. **This is
the other case**: `.apply()` on a *groupby* hands your function a whole sub-table, and a function that
takes a table and returns an answer is exactly the shape we said was right.

In [ ]:
def emissions_per_person(group):
    """Total emissions of a group of rows, per head of its total population."""
    return group["co2_total"].sum() * 1_000_000 / group["population"].sum()


year_2023 = countries_only[countries_only["year"] == 2023]

year_2023.groupby("region").apply(emissions_per_person).round(2)

Seven regions, one number each, computed by a function you could read out loud. Compare it against
the naive version - the plain average of the countries in each region.

In [ ]:
comparison = pd.DataFrame({
    "per_person": year_2023.groupby("region").apply(emissions_per_person),
    "mean_of_countries": year_2023.groupby("region")["co2_pc"].mean(),
})

comparison.round(2)

Mostly close, and then one row that is not: the Middle East and North Africa comes out at **4.09 per
person against an unweighted mean of 9.21**, more than double. Both numbers are arithmetically
correct. The unweighted one is being dragged up by Qatar, Bahrain and Kuwait - small populations,
enormous emissions per head - each counting exactly as much as Pakistan's 248 million people.

Which is right depends entirely on the question. "What does the average country in this region look
like?" wants one; "how much does this region emit per head?" wants the other. The important thing is
knowing that you chose.

> 💡 **Tip:** The rule from last time is unchanged - reach for `np.where`, `np.select` or
plain column arithmetic first, because they are shorter and faster. Reach for `.apply()` when the
calculation genuinely needs the whole group, or the whole row, at once.

## Putting it together

Last time we ended with a function that turned a file into a table. It came with an admission: it
could not remove the aggregates, because the information was in a second file.

It is not an admission any more.

In [ ]:
def load_panel(emissions_path, info_path, countries_only=True):
    """
    Read the emissions and country files, join them, and return one table.

    Parameters
    ----------
    emissions_path : str
        Path to the emissions CSV file.
    info_path : str
        Path to the country lookup CSV file.
    countries_only : bool, optional
        Whether to drop the World Bank aggregate entities such as World and
        Arab World. Default is True, which is what you want for any total,
        average or ranking across entities.

    Returns
    -------
    DataFrame
        One row per entity per year, with emissions per person, the region
        and the income level.
    """
    emissions = load_emissions(emissions_path)
    info = pd.read_csv(info_path)

    # code is the only key in these files that is clean on both sides: two of
    # the names in the lookup file carry a trailing space.
    panel = emissions.merge(
        info[["code", "region", "incomeLevel"]],
        on="code",
        how="left",
        validate="many_to_one",
    )

    if len(panel) != len(emissions):
        raise ValueError("the merge changed the row count")

    if countries_only:
        panel = panel[panel["region"] != "Aggregates"]

    return panel.reset_index(drop=True)


world = load_panel("../data/co2_emissions.csv", "../data/country_info.csv")

print(world.shape)
world.head(3)

One line to call, every decision written down once, and a check inside it that fails loudly rather
than returning a table that is quietly the wrong size. `raise` is how you refuse to return an answer
you do not believe - the same instinct as the errors we deliberately triggered earlier, pointed at
your own code.

That is the shape every analysis worth repeating has: **a function that takes paths and returns a
table you can defend.**

### And a look at what it bought us

In [ ]:
regional = world.groupby(["region", "year"])["co2_total"].sum().reset_index()
east_asia = regional[regional["region"] == "East Asia & Pacific"]

east_asia.plot(x="year", y="co2_total", title="East Asia & Pacific: total CO₂, Mt")

One line, and a picture of the single largest change in global emissions this century. We come back
to how to make figures worth showing somebody.

## Additional resources

- [Group by: split-apply-combine](https://pandas.pydata.org/docs/user_guide/groupby.html) - the
  official account, including the full list of aggregation and transformation methods
- [Merge, join, concatenate and compare](https://pandas.pydata.org/docs/user_guide/merging.html) -
  every way of putting two tables together, with diagrams
- [Time series and date functionality](https://pandas.pydata.org/docs/user_guide/timeseries.html) -
  long, but the tables of `.dt` properties and of `resample` frequency strings are worth bookmarking
- [Working with text data](https://pandas.pydata.org/docs/user_guide/text.html) - the whole `.str`
  accessor, which has a version of nearly every string method you know
- [SKL401](https://isabelhovdahl.github.io/skl401/) - lesson 3.7 covers grouping and aggregation, 3.8
  joining, and 3.9 dates

**Next week:** turning these tables into figures - what makes a chart readable, and how to write one
function that draws the same plot for any country you hand it.